# Preprocessing NLP Dataset ICAR


In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

INPUT_PATH = Path("/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Text_Extracted.json")
OUTPUT_JSONL = Path("/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Text_Preprocessed.jsonl")
OUTPUT_SUMMARY = Path("/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Text_Preprocessing_Summary.json")
OUTPUT_TOP_WORDS = Path("/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Top_Words.csv")

In [2]:
STOPWORDS = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at",
    "be", "because", "been", "before", "being", "below", "between", "both", "but", "by",
    "can", "could",
    "did", "do", "does", "doing", "down", "during",
    "each",
    "few", "for", "from", "further",
    "had", "has", "have", "having", "he", "her", "here", "hers", "herself", "him", "himself", "his", "how",
    "i", "if", "in", "into", "is", "it", "its", "itself",
    "just",
    "me", "more", "most", "my", "myself",
    "no", "nor", "not", "now",
    "of", "off", "on", "once", "only", "or", "other", "our", "ours", "ourselves", "out", "over", "own",
    "s", "same", "she", "should", "so", "some", "such",
    "t", "than", "that", "the", "their", "theirs", "them", "themselves", "then", "there", "these", "they", "this", "those", "through", "to", "too",
    "under", "until", "up",
    "very",
    "was", "we", "were", "what", "when", "where", "which", "while", "who", "whom", "why", "will", "with", "you", "your", "yours", "yourself", "yourselves"
}

In [3]:
def basic_clean(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"---\s*Page\s*\d+\s*---", " ", text, flags=re.IGNORECASE)

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", text)

    text = text.lower()

    text = re.sub(r"[^a-z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
def tokenize_and_filter(cleaned_text: str) -> list[str]:
    tokens = cleaned_text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

In [5]:
with INPUT_PATH.open("r", encoding="utf-8") as f:
    raw_json = f.read()

try:
    data = json.loads(raw_json)
except json.JSONDecodeError as e:
    print(f"JSON standar gagal dibaca: {e}")
    print("Mencoba perbaikan escape character yang tidak valid...")

    repaired = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', raw_json)

    repaired = re.sub(r'\\u(?![0-9a-fA-F]{4})', r'\\\\u', repaired)

    data = json.loads(repaired)
    print("Berhasil dibaca setelah perbaikan escape.")

doc_count = 0
total_raw_chars = 0
total_clean_chars = 0
total_tokens = 0
vocab_counter = Counter()

In [6]:
with OUTPUT_JSONL.open("w", encoding="utf-8") as out:
    for category, docs in data.items():
        if not isinstance(docs, dict):
            continue

        for file_name, raw_text in docs.items():
            if not isinstance(raw_text, str):
                continue

            cleaned_text = basic_clean(raw_text)
            tokens = tokenize_and_filter(cleaned_text)

            record = {
                "category": category,
                "file_name": file_name,
                "raw_char_count": len(raw_text),
                "clean_char_count": len(cleaned_text),
                "token_count": len(tokens),
                "cleaned_text": cleaned_text,
                "tokens": tokens,
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")

            doc_count += 1
            total_raw_chars += len(raw_text)
            total_clean_chars += len(cleaned_text)
            total_tokens += len(tokens)
            vocab_counter.update(tokens)

print(f"Dokumen diproses: {doc_count}")

Dokumen diproses: 159


In [7]:
summary = {
    "input_file": str(INPUT_PATH),
    "output_jsonl": str(OUTPUT_JSONL),
    "documents_processed": doc_count,
    "total_raw_characters": total_raw_chars,
    "total_clean_characters": total_clean_chars,
    "total_tokens_after_filter": total_tokens,
    "unique_tokens": len(vocab_counter),
    "avg_tokens_per_document": round(total_tokens / doc_count, 2) if doc_count else 0,
}

with OUTPUT_SUMMARY.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

{'input_file': '/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Text_Extracted.json',
 'output_jsonl': '/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Text_Preprocessed.jsonl',
 'documents_processed': 159,
 'total_raw_characters': 47939541,
 'total_clean_characters': 42905452,
 'total_tokens_after_filter': 4372792,
 'unique_tokens': 107073,
 'avg_tokens_per_document': 27501.84}

In [8]:
with OUTPUT_TOP_WORDS.open("w", encoding="utf-8") as f:
    f.write("token,frequency\n")
    for token, freq in vocab_counter.most_common(200):
        f.write(f"{token},{freq}\n")

print("File top words berhasil dibuat:", OUTPUT_TOP_WORDS)

File top words berhasil dibuat: /home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Top_Words.csv


In [9]:
with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
    for i in range(3):
        print(f"[JSONL {i+1}]", f.readline().strip()[:400], "...\n")

with OUTPUT_SUMMARY.open("r", encoding="utf-8") as f:
    summary_loaded = json.load(f)

print("Summary:")
print(json.dumps(summary_loaded, ensure_ascii=False, indent=2))

try:
    import pandas as pd
    display(pd.read_csv(OUTPUT_TOP_WORDS).head(10))
except Exception:
    with OUTPUT_TOP_WORDS.open("r", encoding="utf-8") as f:
        for _ in range(10):
            print(f.readline().strip())

[JSONL 1] {"category": "Annual Reports", "file_name": "anrep-02003.pdf", "raw_char_count": 957731, "clean_char_count": 869363, "token_count": 86412, "cleaned_text": "web url dare icar annual report department of agricultural research indian council of and education agricultural research ministry of agriculture new delhi government of india indian council of agricultural research president shri ajit singh ag ...

[JSONL 2] {"category": "Annual Reports", "file_name": "AR-2019-20.pdf", "raw_char_count": 911138, "clean_char_count": 823272, "token_count": 86150, "cleaned_text": "web url indian council df agricultvral research institutes bureaux national research centres and directorates inbluer he mune riltlm wi firtuah dym uuha puaa hitok wuunkyeum idie lun ftztuh he vrta tun hoju iar lntulg maeav ciot heeu korrurch ca ...

[JSONL 3] {"category": "Annual Reports", "file_name": "DARE-Annual-Report-2017-18.pdf", "raw_char_count": 853483, "clean_char_count": 773642, "token_count": 79711, "cle